# Factor Expression Playground

This notebook provides a template for ad-hoc factor research and validation.

## Quick Start

1. Configure parameters in the next cell
2. Run all cells to evaluate your factor expression
3. Inspect results, plots, and summary statistics

## Use Cases

- Validate new factor expressions
- Test NaN handling semantics
- Compare factor behavior across instruments
- Generate factor value series for external analysis

In [29]:
# Configuration Parameters
# Modify these to test different scenarios

# Factor expression (supports full factorexp DSL)
# Note: Use $ prefix for field references (e.g., $close, $volume, $amt)
FULL_EXPRESSION = "Clip(ZScore(TS_Std(When(Greater(TS_Delta($volume, 1), 0), Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1)), Div(0, 0)), 96), 5760), -2, 2)"

# Simpler example for testing:
# EXPRESSION = "TS_Mean($close, 20)"  # 20-bar moving average
# EXPRESSION = "ZScore(TS_Std($close, 20), 5760)"  # Rolling volatility

# Instrument selection
INSTRUMENT_ID = "BTCUSDT.BINANCE"

# Date range
START_DATE = "2022-01-01"
END_DATE = "2022-04-01"

ZSCORE_PERIOD = 5760  # Number of bars for ZScore normalization (e.g., 5760 for 1 day of minute bars)
STD_PERIOD = 20       # Period for standard deviation calculation


In [36]:
STD_EXPRESSION = "TS_Std(When(Greater(TS_Delta($volume, 1), 0), Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1)), Div(0, 0)), 96)"
WHEN_EXPRESSION = "When(Greater(TS_Delta($volume, 1), 0), Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1)), Div(0, 0))"
GREATER_EXPRESSION = "Greater(TS_Delta($volume, 1), 0)"
TRUE_EXPRESSION = "Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1))"
SUB_EXPRESSION = "Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1))"
DENO_EXPRESSION = "TS_Ref(Div($amt, $volume), 1)"
VWAP_EXPRESSION = "Div($amt, $volume)"
AMT_EXPRESSION = "$amt"
VOLUME_EXPRESSION = "$volume"

In [31]:
# Setup environment
import sys
from pathlib import Path

# Add parent directory to path for imports
notebook_dir = Path.cwd()
research_dir = notebook_dir.parent
backtest_dir = research_dir.parent

if str(backtest_dir) not in sys.path:
    sys.path.insert(0, str(backtest_dir))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from research.utils import (
    run_expression,
    FactorRequest,
    get_available_instruments,
)

# Plotting setup
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Environment setup complete")

✅ Environment setup complete


## 1. Available Instruments

List all instruments available in the catalog.

In [32]:
# List available instruments
instruments = get_available_instruments()
print(f"Found {len(instruments)} instruments in catalog\n")
print("First 20 instruments:")
for inst in instruments[:20]:
    print(f"  - {inst}")

if INSTRUMENT_ID not in instruments:
    print(f"\n⚠️  Warning: {INSTRUMENT_ID} not found in catalog")
    print(f"Available instruments: {', '.join(instruments[:10])}...")
else:
    print(f"\n✅ {INSTRUMENT_ID} is available")

Found 40 instruments in catalog

First 20 instruments:
  - 1000FLOKIUSDT.BINANCE
  - 1000PEPEUSDT.BINANCE
  - 1000SHIBUSDT.BINANCE
  - AAVEUSDT.BINANCE
  - ADAUSDT.BINANCE
  - ALGOUSDT.BINANCE
  - APTUSDT.BINANCE
  - ARKMUSDT.BINANCE
  - ARUSDT.BINANCE
  - AVAXUSDT.BINANCE
  - AXSUSDT.BINANCE
  - BCHUSDT.BINANCE
  - BNBUSDT.BINANCE
  - BTCUSDT.BINANCE
  - CHZUSDT.BINANCE
  - DOGEUSDT.BINANCE
  - DYDXUSDT.BINANCE
  - EOSUSDT.BINANCE
  - ETCUSDT.BINANCE
  - ETHUSDT.BINANCE

✅ BTCUSDT.BINANCE is available


## 2. Factor Evaluation

Run the factor expression and collect results.

In [37]:
# Create request
print("\nCreating factor request...")

full_request = FactorRequest(
    expression=FULL_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=ZSCORE_PERIOD,
)

std_request = FactorRequest(
    expression=STD_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)

# add more requests as needed
when_request = FactorRequest(
    expression=WHEN_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)

greater_request = FactorRequest(
    expression=GREATER_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)

true_request = FactorRequest(
    expression=TRUE_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)

sub_request = FactorRequest(
    expression=SUB_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)

deno_request = FactorRequest(
    expression=DENO_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)

amt_request = FactorRequest(
    expression=AMT_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)
vwap_request = FactorRequest(
    expression=VWAP_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,
    end_date=END_DATE,
    period=STD_PERIOD,
)
volume_request = FactorRequest(
    expression=VOLUME_EXPRESSION,
    instrument_id=INSTRUMENT_ID,
    start_date=START_DATE,  
    end_date=END_DATE,
    period=STD_PERIOD,
)


Creating factor request...


In [34]:
def evaluate_request(name, request):
    print(f"\nEvaluating expression: {request.expression}\n", flush=True)
    result = run_expression(request)

    print(f"✅ {name}: Generated {len(result)} factor values", flush=True)
    print(f"NaN count: {result['factor_value'].isna().sum()} ({result['factor_value'].isna().sum() / len(result) * 100:.2f}%)")
    display(result.head(10))
    return result

In [38]:
# volume_result = evaluate_request("Volume Expression", volume_request)

# amt_result = evaluate_request("AMOUNT Expression", amt_request)

# deno_result = evaluate_request("DENO Expression", deno_request)

# sub_result = evaluate_request("SUB Expression", sub_request)

# true_result = evaluate_request("True Expression", true_request)

# full_result = evaluate_request("Full Expression", full_request)

std_result = evaluate_request("STD Expression", std_request)


Evaluating expression: TS_Std(When(Greater(TS_Delta($volume, 1), 0), Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1)), Div(0, 0)), 96)

Loading bar data...
Loading data from catalog at /Users/arandott/Desktop/documents_for_work/Chaochien/nautilus_trader/factorexp_backtest/catalog
Catalog instruments: []
Loading bars for BTCUSDT.BINANCE from 2022-01-01 to 2022-04-01
Loaded 8631 bars
Initializing FactorExpIndicator...
Initialized FactorExpIndicator with expression: TS_Std(When(Greater(TS_Delta($volume, 1), 0), Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1)), Div(0, 0)), 96)
Processing bars:  75%|███████▍  | 6441/8631 [00:03<00:01, 1361.58bar/s]

Processing bars: 100%|██████████| 8631/8631 [00:04<00:00, 1791.01bar/s]
✅ STD Expression: Generated 8535 factor values
NaN count: 0 (0.00%)


,ts_event,ts_init,factor_value,bar_close,bar_volume
0,2022-01-02 00:00:00,2022-01-02 00:00:00,0.003259,47501.49,2253.792
1,2022-01-02 00:15:00,2022-01-02 00:15:00,0.003258,47541.16,2638.528
2,2022-01-02 00:30:00,2022-01-02 00:30:00,0.003237,47640.00,2335.877
3,2022-01-02 00:45:00,2022-01-02 00:45:00,0.003237,47622.14,1003.086
4,2022-01-02 01:00:00,2022-01-02 01:00:00,0.003222,47446.24,1383.171
5,2022-01-02 01:15:00,2022-01-02 01:15:00,0.003220,47453.88,1048.257
6,2022-01-02 01:30:00,2022-01-02 01:30:00,0.003194,47389.27,1214.045
7,2022-01-02 01:45:00,2022-01-02 01:45:00,0.003194,47381.20,751.360
8,2022-01-02 02:00:00,2022-01-02 02:00:00,0.003193,47436.14,1363.015
9,2022-01-02 02:15:00,2022-01-02 02:15:00,0.003231,47330.20,992.756


In [55]:
# read /Users/arandott/Desktop/documents_for_work/Chaochien/nautilus_trader/factorexp_backtest/data/demo_factor.parquet
bck = pd.read_csv('/Users/arandott/Desktop/documents_for_work/Chaochien/nautilus_trader/factorexp_backtest/data/demo_factor.parquet')
bck = bck.iloc[288:(8343 + 288), 14:15]
bck

,13
288,0.002665
289,0.002644
290,0.002663
291,0.002671
292,0.002693
...,...
8626,0.002534
8627,0.002497
8628,0.002497
8629,0.002474


In [79]:
# read /Users/arandott/Desktop/documents_for_work/Chaochien/nautilus_trader/factorexp_backtest/data/demo_factor.parquet
bck = pd.read_csv('/Users/arandott/Desktop/documents_for_work/Chaochien/nautilus_trader/factorexp_backtest/data/demo_factor.parquet')
bck = bck.iloc[288:(8343 + 288), 14:15].reset_index(drop=True)
factorexp_NT = std_result.iloc[(96 + 96):, :].reset_index(drop=True)
# merge the two dataframes on their index
merged = pd.merge(factorexp_NT, bck, left_index=True, right_index=True, suffixes=('_factorexp', '_bck'))
merged.rename(columns={'13': 'factor_value_bck'}, inplace=True)
merged["relative_diff"] = (merged['factor_value'] - merged['factor_value_bck']).abs() / merged['factor_value'].abs()
# compare(a - a.shift(1), b - b.shift(1))
merged["a_diff"] = merged['factor_value'] - merged['factor_value'].shift(1)
merged["b_diff"] = merged['factor_value_bck'] - merged['factor_value_bck'].shift(1)
merged


,ts_event,ts_init,factor_value,bar_close,bar_volume,factor_value_bck,relative_diff,a_diff,b_diff
0,2022-01-04 00:00:00,2022-01-04 00:00:00,0.002694,46410.00,2204.210,0.002665,0.010694,NaN,NaN
1,2022-01-04 00:15:00,2022-01-04 00:15:00,0.002672,46485.89,3912.244,0.002644,0.010470,-0.000022,-0.000021
2,2022-01-04 00:30:00,2022-01-04 00:30:00,0.002692,46284.61,1533.616,0.002663,0.010695,0.000020,0.000019
3,2022-01-04 00:45:00,2022-01-04 00:45:00,0.002699,46251.50,1867.984,0.002671,0.010470,0.000008,0.000008
4,2022-01-04 01:00:00,2022-01-04 01:00:00,0.002722,46298.00,1332.291,0.002693,0.010694,0.000023,0.000022
...,...,...,...,...,...,...,...,...,...
8338,2022-03-31 23:00:00,2022-03-31 23:00:00,0.002452,45679.70,1268.547,0.002534,0.033538,-0.000022,0.000000
8339,2022-03-31 23:15:00,2022-03-31 23:15:00,0.002427,45672.70,1615.501,0.002497,0.028604,-0.000024,-0.000037
8340,2022-03-31 23:30:00,2022-03-31 23:30:00,0.002434,45419.20,4000.235,0.002497,0.025873,0.000006,0.000000
8341,2022-03-31 23:45:00,2022-03-31 23:45:00,0.002470,45506.00,8936.148,0.002474,0.001439,0.000036,-0.000023


In [72]:
merged.iloc[2000:2020]

,ts_event,ts_init,factor_value,bar_close,bar_volume,factor_value_bck,relative_diff
2000,2022-01-24 20:00:00,2022-01-24 20:00:00,0.006268,36196.75,6647.893,0.006190,0.012578
2001,2022-01-24 20:15:00,2022-01-24 20:15:00,0.006268,36081.15,5843.403,0.006190,0.012578
2002,2022-01-24 20:30:00,2022-01-24 20:30:00,0.006285,36374.59,10164.031,0.006208,0.012270
2003,2022-01-24 20:45:00,2022-01-24 20:45:00,0.006758,37210.71,22912.329,0.006677,0.011976
2004,2022-01-24 21:00:00,2022-01-24 21:00:00,0.006839,36889.99,10808.367,0.006755,0.012270
2005,2022-01-24 21:15:00,2022-01-24 21:15:00,0.006904,36883.35,4402.031,0.006817,0.012578
2006,2022-01-24 21:30:00,2022-01-24 21:30:00,0.006904,36860.36,3981.879,0.006817,0.012578
2007,2022-01-24 21:45:00,2022-01-24 21:45:00,0.006846,36800.00,4026.918,0.006762,0.012269
2008,2022-01-24 22:00:00,2022-01-24 22:00:00,0.006846,36771.79,3570.367,0.006762,0.012269
2009,2022-01-24 22:15:00,2022-01-24 22:15:00,0.006846,36564.64,3197.407,0.006762,0.012269


In [21]:
full_result.head(20)

,ts_event,ts_init,factor_value,bar_close,bar_volume
0,2022-03-17 02:15:00,2022-03-17 02:15:00,0.065994,40994.1,2686.613
1,2022-03-17 02:30:00,2022-03-17 02:30:00,0.069087,41023.7,1464.160
2,2022-03-17 02:45:00,2022-03-17 02:45:00,0.068831,41104.4,1286.740
3,2022-03-17 03:00:00,2022-03-17 03:00:00,0.068572,40989.0,1176.350
4,2022-03-17 03:15:00,2022-03-17 03:15:00,0.068314,41090.0,976.968
5,2022-03-17 03:30:00,2022-03-17 03:30:00,0.066164,41030.6,1998.502
6,2022-03-17 03:45:00,2022-03-17 03:45:00,0.056354,40999.0,1340.869
7,2022-03-17 04:00:00,2022-03-17 04:00:00,0.049866,41028.7,2043.870
8,2022-03-17 04:15:00,2022-03-17 04:15:00,0.049605,40967.8,1169.663
9,2022-03-17 04:30:00,2022-03-17 04:30:00,0.048267,40986.0,1213.540


## 3. Summary Statistics

Analyze factor value distribution and characteristics.

In [27]:
def factor_statistics(result):
    # Basic statistics
    print("Factor Value Statistics:")
    print("=" * 50)
    print(result['factor_value'].describe())
    print("\nAdditional Metrics:")
    print(f"Skewness: {result['factor_value'].skew():.4f}")
    print(f"Kurtosis: {result['factor_value'].kurtosis():.4f}")
    print(f"\nValue Range: [{result['factor_value'].min():.4f}, {result['factor_value'].max():.4f}]")

In [28]:
factor_statistics(std_result)

Factor Value Statistics:
count    7191.000000
mean        0.004059
std         0.000531
min         0.002679
25%         0.003682
50%         0.004022
75%         0.004410
max         0.005137
Name: factor_value, dtype: float64

Additional Metrics:
Skewness: 0.1218
Kurtosis: -0.7557

Value Range: [0.0027, 0.0051]


## 4. Visualization

Plot factor values over time and distribution.

In [ ]:
# Time series plot
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Factor value over time
axes[0].plot(result['ts_event'], result['factor_value'], linewidth=0.8, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
axes[0].set_title(f'Factor Value Over Time: {INSTRUMENT_ID}', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Factor Value')
axes[0].grid(True, alpha=0.3)

# Price reference
ax2 = axes[0].twinx()
ax2.plot(result['ts_event'], result['bar_close'], color='orange', alpha=0.3, linewidth=0.5)
ax2.set_ylabel('Close Price', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

# Distribution
axes[1].hist(result['factor_value'].dropna(), bins=50, alpha=0.7, edgecolor='black')
axes[1].axvline(x=result['factor_value'].mean(), color='red', linestyle='--', 
                label=f'Mean: {result["factor_value"].mean():.4f}')
axes[1].axvline(x=result['factor_value'].median(), color='green', linestyle='--',
                label=f'Median: {result["factor_value"].median():.4f}')
axes[1].set_title('Factor Value Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Factor Value')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. NaN Analysis

Detailed analysis of NaN values and their patterns.

In [ ]:
# Identify NaN sequences
is_nan = result['factor_value'].isna()
nan_transitions = is_nan.astype(int).diff().fillna(0)

# Find consecutive NaN runs
nan_runs = []
run_start = None

for idx, (is_nan_val, transition) in enumerate(zip(is_nan, nan_transitions)):
    if transition == 1:  # Start of NaN run
        run_start = idx
    elif transition == -1 and run_start is not None:  # End of NaN run
        nan_runs.append((run_start, idx - 1))
        run_start = None

# Handle case where NaN run extends to end
if run_start is not None:
    nan_runs.append((run_start, len(is_nan) - 1))

print(f"NaN Analysis:")
print("=" * 50)
print(f"Total NaN values: {is_nan.sum()}")
print(f"Number of NaN runs: {len(nan_runs)}")

if nan_runs:
    run_lengths = [end - start + 1 for start, end in nan_runs]
    print(f"\nLongest NaN run: {max(run_lengths)} bars")
    print(f"Average NaN run length: {np.mean(run_lengths):.2f} bars")
    print(f"\nFirst 5 NaN runs (start_idx, end_idx, length):")
    for i, (start, end) in enumerate(nan_runs[:5]):
        print(f"  Run {i+1}: ({start}, {end}, {end-start+1} bars)")
else:
    print("\n✅ No NaN values found in factor output")

## 6. Export Results

Save factor values for external analysis or sharing.

In [ ]:
# Prepare output directory
output_dir = research_dir / "results"
output_dir.mkdir(exist_ok=True)

# Generate filename with timestamp
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
instrument_clean = INSTRUMENT_ID.replace(".", "_")
output_file = output_dir / f"factor_{instrument_clean}_{timestamp}.feather"

# Save to feather format (fast, compressed)
result.reset_index(drop=True).to_feather(output_file)

print(f"✅ Results saved to: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024:.2f} KB")

## Next Steps

### Modify Expression

Try these example expressions:

```python
# Simple moving average
EXPRESSION = "TS_Mean($close, 20)"

# Rolling volatility (ZScore normalized)
EXPRESSION = "ZScore(TS_Std($close, 20), 5760)"

# VWAP return volatility (requires extended fields)
EXPRESSION = "Clip(ZScore(TS_Std(When(Greater(TS_Delta($volume, 1), 0), Div(Sub(Div($amt, $volume), TS_Ref(Div($amt, $volume), 1)), TS_Ref(Div($amt, $volume), 1)), Div(0, 0)), 20), 5760), -2, 2)"

# Momentum factor
EXPRESSION = "Clip(ZScore(TS_Delta($close, 20), 5760), -2, 2)"
```

### Compare Instruments

Change `INSTRUMENT_ID` to compare behavior:
- `BTCUSDT.BINANCE`
- `ETHUSDT.BINANCE`
- `BNBUSDT.BINANCE`

### Adjust Parameters

- `ZSCORE_PERIOD`: Lookback window for normalization
- `START_DATE` / `END_DATE`: Analysis period

### Use in Backtest

Once validated, copy the expression to `factorexp_backtest/configs/factors.yaml`